In [2]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = yf.download(
    "AAPL",
    start="2010-01-01",
    interval="1d"
)

close_prices = data["Close"].squeeze()

[*********************100%***********************]  1 of 1 completed


In [60]:
import numpy as np

window_size = 50
future_days = 10

X = []
y = []

for start in range(len(close_prices) - window_size - future_days + 1):
    end = start + window_size

    # The 50 days that the model gets to see
    window = close_prices.iloc[start:end]

    # Convert absolute prices into relative movement
    normalized_window = (window / window.iloc[0]) - 1

    # Last price visible to the model
    current_price = close_prices.iloc[end - 1]

    # The NEXT 10 days
    future_window = close_prices.iloc[end:end + future_days]

    # Highest price reached during those 10 days
    max_future_price = future_window.max()

    # Best possible return during those 10 days
    max_return = max_future_price / current_price - 1

    X.append(normalized_window.values)
    y.append(max_return)

X = np.array(X)
y = np.array(y)

print("X:", X.shape)
print("y:", y.shape)

X: (4140, 50)
y: (4140,)


In [70]:
def build_dataset(close_prices, window_size=50, future_days=10):
    X = []

    max_returns = []
    days_to_max = []

    window_end_dates = []
    max_dates = []

    number_of_examples = (len(close_prices) - window_size - future_days + 1)

    for start in range(number_of_examples):
        end = start + window_size

        # -------------------------
        # PAST - visible to model
        # -------------------------

        window = close_prices.iloc[start:end]

        #normalize the values
        normalized_window = (
            (window / window.iloc[0]) - 1
        )

        # -------------------------
        # FUTURE - hidden from model
        # -------------------------

        future_window = close_prices.iloc[end:end + future_days]

        #normalize the returns as well
        future_returns = (
            (future_window / window.iloc[-1]) - 1
        )

        # Position & value of the highest return
        max_position = np.argmax(future_returns.values)

        max_value_return = future_returns.iloc[max_position]

        # trading day after entry"
        days_until_max = max_position + 1

        # -------------------------
        # Save this example
        # -------------------------

        X.append(normalized_window.values)

        max_returns.append(max_value_return)
        days_to_max.append(days_until_max)

        window_end_dates.append(window.index[-1])
        max_dates.append(
            future_window.index[max_position]
        )

    X = np.array(X)

    targets = pd.DataFrame({
        "window_end_date": window_end_dates,
        "max_value_return": max_returns,
        "days_to_max": days_to_max,
        "max_date": max_dates
    })

    return X, targets

In [71]:
X, targets = build_dataset(
    close_prices,
    window_size=50,
    future_days=10
)

print(X.shape)
print(targets.shape)

print(targets[:10])

(4140, 50)
(4140, 4)
  window_end_date  max_value_return  days_to_max   max_date
0      2010-03-16          0.050791           10 2010-03-30
1      2010-03-17          0.052338            9 2010-03-30
2      2010-03-18          0.050389           10 2010-04-01
3      2010-03-19          0.073071           10 2010-04-05
4      2010-03-22          0.065806           10 2010-04-06
5      2010-03-23          0.053600           10 2010-04-07
6      2010-03-24          0.048960            9 2010-04-07
7      2010-03-25          0.066799           10 2010-04-09
8      2010-03-26          0.049328           10 2010-04-12
9      2010-03-29          0.043203           10 2010-04-13
